In [ ]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px

In [ ]:
# Read distance matrix
dist_mat = pd.read_csv("../output/handshape_distance_matrix.csv")

dist_mat.head()

In [ ]:
# Top 10 most similar handshapes to 'fbx\M20241107_6263_260710_2.fbx_segment_2'

# Get the row for the specified handshape
# Get file name from index 2
file_name = dist_mat.index[2]
handshape_row = dist_mat.iloc[2]

# Get the indices of the 10 smallest distances
handshape_distances = handshape_row[1:].to_numpy()
similar_indices = handshape_distances.argsort()

# Get the names of the 10 most similar handshapes (excluding any names starting with 'fbx\')
total = 0
similar_handshape_names = []
for index in similar_indices:
    if total >= 10:
        break
    handshape_name = dist_mat.columns[index + 1]  # +1 to account for the 'Name' column
    if not handshape_name.startswith('fbx\\'):
        similar_handshape_names.append(handshape_name)
        total += 1

print("10 most similar handshapes to:")
for handshape in similar_handshape_names:
    print(handshape)

In [ ]:
# Get distance array
dist_mat = dist_mat.drop(columns=['Name'])
distance_array = dist_mat.to_numpy()

In [ ]:
# # Make all distances 0 or higher
# distance_array += np.abs(np.min(distance_array))

In [ ]:
# t-SNE on the distance matrix
random_num = random.randint(0, 10000)
# random_num = 8079  # This is the last one I used, so for reproducibility hardcoded
print("Using random seed:", random_num)
tsne_dist = manifold.TSNE(n_components=2, perplexity=50, learning_rate=200, init='random', metric='precomputed', method='barnes_hut', random_state=random_num)
X_tsne_dist = tsne_dist.fit_transform(distance_array)

In [ ]:
# Dataframe for the px plot
tsne_df = pd.DataFrame({
    'tsne_1': X_tsne_dist[:, 0],
    'tsne_2': X_tsne_dist[:, 1],
    'name': dist_mat.columns,
    'is_handshape': [name.startswith("../") for name in dist_mat.columns]
})

# Plot
fig = px.scatter(
    tsne_df,
    x='tsne_1',
    y='tsne_2',
    color='is_handshape',
    hover_data=['name'],
    title="t-SNE Dimensionality Reduction of the Shape Features"
)

fig.update_layout(width=800, height=800)
fig.show()